In [ ]:
import yfinance as yf
import pandas as pd
import os

In [ ]:
proxy = 'http://127.0.0.1:7890'  # 代理设置，此处修改
os.environ['HTTP_PROXY'] = proxy
os.environ['HTTPS_PROXY'] = proxy

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ======================
# 0. 全局画图配置：解决中文乱码，负号显示
# ======================
plt.rcParams['font.sans-serif'] = ['SimHei']   # 可换成 ['Microsoft YaHei'] / ['PingFang SC'] 等
plt.rcParams['axes.unicode_minus'] = False

# ======================
# 1. 读入数据
# ======================
# ======================
df = pd.read_csv("../data/四个板块.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date").sort_index()



# ======================
# 2. 定义 4 个板块，大/中/小 12 个组合
# ======================
AI_max = ["MSFT", "GOOGL", "META", "AMZN", "TSLA", "ORCL", "CRM", "ADBE"] 
AI_mid = ["SNOW", "PLTR", "DDOG", "MDB", "U", "CRWD", "ZS", "NET"]
AI_min = ["AI", "BBAI", "SOUN", "VERI", "UPST", "CXAI"]

finance_max = ["JPM", "BAC", "WFC", "C", "GS", "MS", "AXP", "BLK", "SCHW"]
finance_mid = ["USB", "PNC", "TFC", "COF", "STT", "BK", "MTB", "FITB", "CFG"]
finance_min = ["HOOD", "SOFI", "AFRM", "LC", "MQ", "NU", "HIPO", "ROOT"]

encryption_max = ["COIN", "MSTR", "MARA", "RIOT", "CLSK", "HUT", "HIVE", "BITF"]
encryption_mid = ["CIFR", "WULF", "IREN", "BTBT", "CORZ", "BKKT", "BTDR"]
encryption_min = ["ANY", "ARBK", "ABTC", "GREE", "BTCS"]

semiconductor_max = ["NVDA", "AMD", "INTC", "AVGO", "QCOM", "TXN", "ADI", "MU", "NXPI", "TSM"]
semiconductor_mid = ["MCHP", "MPWR", "ON", "LRCX", "KLAC", "TER", "GFS"]
semiconductor_min = ["AEHR", "NVTS", "INDI", "LASR", "HIMX", "SMTC", "AMBA", "POWI"]

groups = {
    "AI_max": AI_max,
    "AI_mid": AI_mid,
    "AI_min": AI_min,
    "Fin_max": finance_max,
    "Fin_mid": finance_mid,
    "Fin_min": finance_min,
    "Cryp_max": encryption_max,
    "Cryp_mid": encryption_mid,
    "Cryp_min": encryption_min,
    "Semi_max": semiconductor_max,
    "Semi_mid": semiconductor_mid,
    "Semi_min": semiconductor_min,
}

# 英文组合名 → 中文名，用于图例
name_map = {
    "AI_max": "AI蓝筹",
    "AI_mid": "AI中型",
    "AI_min": "AI小盘",

    "Fin_max": "金融蓝筹",
    "Fin_mid": "金融中型",
    "Fin_min": "金融小盘",

    "Cryp_max": "加密蓝筹",
    "Cryp_mid": "加密中型",
    "Cryp_min": "加密小盘",

    "Semi_max": "半导体蓝筹",
    "Semi_mid": "半导体中型",
    "Semi_min": "半导体小盘",
}

# ======================
# 3. ETF 组合（分母）
# ======================
ETF = ["XLP", "XLU", "AGG"]
ETF_cols = [c for c in ETF if c in df.columns]
if len(ETF_cols) == 0:
    raise ValueError("ETF 列一个都没找到，请检查 CSV 是否包含 XLP / XLU / AGG")

etf_price = df[ETF_cols].mean(axis=1)

# ======================
# 4. 选择基准日：2025-09-01 之后的第一个有数据的日期
# ======================
start_date = pd.Timestamp("2025-09-01")
valid_dates = df.index[df.index >= start_date]
if len(valid_dates) == 0:
    raise ValueError("没有 2025-09-01 之后的数据")
base_date = valid_dates[0]

etf_index = etf_price / etf_price.loc[base_date]

# ======================
# 5. 计算 12 个组合相对 ETF 的指数（起点=100）
# ======================
result = pd.DataFrame(index=df.index)

for name, ticker_list in groups.items():
    cols = [c for c in ticker_list if c in df.columns]
    if len(cols) == 0:
        print(f"{name}: 在数据中一个股票都没有，跳过")
        continue

    if len(cols) < len(ticker_list):
        missing = set(ticker_list) - set(cols)
        print(f"{name}: 有缺失股票列，被自动忽略: {sorted(missing)}")

    group_price = df[cols].mean(axis=1)
    group_index = group_price / group_price.loc[base_date]
    rel_index = group_index / etf_index
    result[name] = rel_index * 100

result = result[result.index >= base_date]

# ======================
# 6. 按板块分成四个子图
# ======================
# 每个板块包含的组合
sector_groups = {
    "AI板块": ["AI_max", "AI_mid", "AI_min"],
    "金融板块": ["Fin_max", "Fin_mid", "Fin_min"],
    "加密板块": ["Cryp_max", "Cryp_mid", "Cryp_min"],
    "半导体板块": ["Semi_max", "Semi_mid", "Semi_min"],
}

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
axes = axes.flatten()  # 简化索引：0,1,2,3

for ax, (sector_name, group_names) in zip(axes, sector_groups.items()):
    for g in group_names:
        if g not in result.columns:
            continue
        label = name_map.get(g, g)
        ax.plot(result.index, result[g], label=label)
    ax.axhline(100, linestyle="--", linewidth=1)
    ax.set_title(f"{sector_name}（基准日：{base_date.date()}）")
    ax.set_ylabel("相对 ETF 指数（基准=100）")
    ax.legend(fontsize=9)

# 底部统一 x 轴标签
fig.text(0.5, 0.04, "日期", ha="center")

plt.tight_layout(rect=[0.03, 0.05, 0.97, 0.98])
plt.show()
